In [4]:
# Set your username here - use it consistently across all resources
USERNAME = "petboga"

In [3]:
import datetime
import json

import boto3
import requests

In [6]:
# Try different dates to see how the data changes
DATE_PARAM = "2025-11-28"

date = datetime.datetime.strptime(DATE_PARAM, "%Y-%m-%d")

# Construct the API URL
url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/{date.strftime('%Y/%m/%d')}"
print(f"Requesting REST API URL: {url}")

# Make the API request
wiki_server_response = requests.get(url, headers={"User-Agent": "curl/7.68.0"})
wiki_response_status = wiki_server_response.status_code
wiki_response_body = wiki_server_response.text

print(f"Wikipedia REST API Response body: {wiki_response_body[:500]}...")
print(f"Wikipedia REST API Response Code: {wiki_response_status}")

# Validate response
if wiki_response_status != 200:
    raise Exception(f"Received non-OK status code from Wiki Server: {wiki_response_status}")
print(f"Successfully retrieved Wikipedia data, content-length: {len(wiki_response_body)}")

Requesting REST API URL: https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/2025/11/28
Wikipedia REST API Response body: {"items":[{"project":"en.wikipedia","access":"all-access","year":"2025","month":"11","day":"28","articles":[{"article":"Main_Page","views":6029837,"rank":1},{"article":"Special:Search","views":780156,"rank":2},{"article":"Stranger_Things_season_5","views":506990,"rank":3},{"article":"Stranger_Things","views":292672,"rank":4},{"article":"Google_Chrome","views":289193,"rank":5},{"article":"Wikipedia:Featured_pictures","views":249961,"rank":6},{"article":"Millie_Bobby_Brown","views":158031,"rank":7...
Wikipedia REST API Response Code: 200
Successfully retrieved Wikipedia data, content-length: 55686


In [16]:
# Parse the API response and extract top edits
wiki_response_parsed = wiki_server_response.json()
# most_viewed = wiki_response_parsed["items"][0]["articles"][0]["rank"]
most_viewed = wiki_response_parsed["items"][0]["articles"]
# top_edits = wiki_response_parsed["items"][0]["results"][0]["top"]

# Transform to JSON Lines format
current_time = datetime.datetime.now(datetime.timezone.utc)
json_lines = ""
for page in most_viewed[:5]:
    record = {
        "title": page["article"],
        "views": page["views"],
        "date": date.strftime("%Y-%m-%d"),
        "retrieved_at": current_time.replace(tzinfo=None).isoformat(),
    }
    json_lines += json.dumps(record) + "\n"

print(f"Transformed {len(most_viewed)} records to JSON Lines")
print(f"First few lines:\n{json_lines[:500]}...")

print(len(most_viewed))

Transformed 1000 records to JSON Lines
First few lines:
{"title": "Main_Page", "views": 6029837, "date": "2025-11-28", "retrieved_at": "2025-12-10T16:34:58.719324"}
{"title": "Special:Search", "views": 780156, "date": "2025-11-28", "retrieved_at": "2025-12-10T16:34:58.719324"}
{"title": "Stranger_Things_season_5", "views": 506990, "date": "2025-11-28", "retrieved_at": "2025-12-10T16:34:58.719324"}
{"title": "Stranger_Things", "views": 292672, "date": "2025-11-28", "retrieved_at": "2025-12-10T16:34:58.719324"}
{"title": "Google_Chrome", "views": 28919...
1000
